In [5]:
import os
import glob
import string
import unicodedata
import torch
import string
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
import torch.optim as optim
from torch import nn
import torch.nn.functional as F
from collections import Counter

# Function to convert a Unicode string to plain ASCII
def unicode_to_ascii(s):
    return ''.join(
        c for c in unicodedata.normalize('NFD', s)
        if unicodedata.category(c) != 'Mn'
        and c in all_letters
    )

# Read lines from a file and convert to ASCII
def read_lines(filename):
    with open(filename, encoding='utf-8') as file:
        lines = file.read().strip().split('\n')
    return [unicode_to_ascii(line) for line in lines]

# Constants
all_letters = string.ascii_letters + " .,;'"
n_letters = len(all_letters)

# Find letter index from all_letters, e.g. "a" = 0
def letter_to_index(letter):
    return all_letters.find(letter)

# Turn a line into a <line_length x 1 x n_letters> tensor
def line_to_tensor(line):
    tensor = torch.zeros(len(line),n_letters)
    for li, letter in enumerate(line):
        tensor[li][letter_to_index(letter)] = 1
    return tensor
def extractCategory(filename):
    return os.path.basename(filename).split('.')[0]

def labelToIndex(categories):
 return {category:i for i ,category in enumerate(categories)}

def labelToTensor(label,n_labels):
    tensor = torch.zeros(n_labels)
    tensor[label] = 1
    return tensor

def outputTensor(outputs,output_index_dict):
    n_labels = len(output_index_dict)
    output_tensor = [labelToTensor(output_index_dict[output],n_labels) for output in outputs]
    return output_tensor

def inputTensor(input):
    tensors = []
    for i,name in enumerate(input):
        name_tensor = line_to_tensor(name)
        tensors.append(name_tensor)

    input_tensor = pad_sequence(tensors,batch_first= True)

    return input_tensor

In [9]:
data_path = 'data/names/*.txt'
filenames = glob.glob(data_path)
categories = list()
data = list()
labels = list()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
for filename in filenames:
    category = os.path.basename(filename).split('.')[0]
    categories.append(category)
    lines = read_lines(filename)
    for line in lines:
        data.append(line)
        labels.append(category)

### Output Tensor ###
label_to_index_dict = labelToIndex(categories)
label_tensor = torch.stack(outputTensor(labels,label_to_index_dict))

### Input Tensor ###
data_tensor = inputTensor(data)

In [10]:
class NamesDataset(Dataset):
    def __init__(self,data_tensor,label_tensor):
        self.data_tensor = data_tensor
        self.label_tensor = label_tensor

    def __len__(self):
        return len(self.data_tensor)
    
    def __getitem__(self, index):
        return self.data_tensor[index],self.label_tensor[index]
    
dataset = NamesDataset(data_tensor, label_tensor)

In [11]:
counter = Counter(labels)
total_count = sum(counter.values())
class_weights = [total_count / counter[label] for label in label_to_index_dict]
class_weights = torch.tensor(class_weights, dtype=torch.float).to(device)

In [14]:
class SimpleRNN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, num_layers):
        super(SimpleRNN, self).__init__()
        self.rnn = nn.RNN(input_size, hidden_size,num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        out, _ = self.rnn(x)
        out = self.fc(out[:, -1, :])  # Use the output of the last time step
        return out
    
hidden_size = 256
num_layers = 4

model = SimpleRNN(n_letters, hidden_size, len(label_to_index_dict), num_layers).to(device)


In [17]:
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = optim.SGD(model.parameters(), lr=0.005)

# Training loop
num_epochs = 100

batch_size = 8  # Adjust batch size as needed

# Create DataLoader
data_loader = DataLoader(dataset, batch_size=batch_size,shuffle=True)

In [18]:
best_loss = float('inf')
path = f'RNN_{num_layers}_layer.pt'

for epoch in range(num_epochs):
    for batch_data, batch_labels in data_loader:
        # Forward pass
        batch_data, batch_labels = batch_data.to(device), batch_labels.to(device)
        outputs = model(batch_data)
        loss = criterion(outputs, batch_labels.argmax(dim=1))

        # Backward pass and optimization
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}')
    if loss.item() < best_loss:
        best_loss = loss.item()
        torch.save(model.state_dict(),path)

Epoch [1/100], Loss: 2.1539
Epoch [2/100], Loss: 2.3333
Epoch [3/100], Loss: 3.1534
Epoch [4/100], Loss: 2.6593
Epoch [5/100], Loss: 3.9093
Epoch [6/100], Loss: 2.9526
Epoch [7/100], Loss: 3.1811
Epoch [8/100], Loss: 2.3187
Epoch [9/100], Loss: 2.3007
Epoch [10/100], Loss: 2.2897
Epoch [11/100], Loss: 2.2656
Epoch [12/100], Loss: 3.0013
Epoch [13/100], Loss: 2.0373
Epoch [14/100], Loss: 3.2893
Epoch [15/100], Loss: 2.9044
Epoch [16/100], Loss: 3.2006
Epoch [17/100], Loss: 2.7807
Epoch [18/100], Loss: 2.1818
Epoch [19/100], Loss: 0.7314
Epoch [20/100], Loss: 2.9976
Epoch [21/100], Loss: 2.4177
Epoch [22/100], Loss: 2.7693
Epoch [23/100], Loss: 3.1803
Epoch [24/100], Loss: 2.2133
Epoch [25/100], Loss: 1.0543
Epoch [26/100], Loss: 1.9032
Epoch [27/100], Loss: 1.0596
Epoch [28/100], Loss: 0.8986
Epoch [29/100], Loss: 0.8600
Epoch [30/100], Loss: 0.8374
Epoch [31/100], Loss: 2.7826
Epoch [32/100], Loss: 1.4372
Epoch [33/100], Loss: 0.1319
Epoch [34/100], Loss: 4.0549
Epoch [35/100], Loss: 0

In [19]:
num_layers = 4
path = f'RNN_{num_layers}_layer.pt'
model = SimpleRNN(n_letters, hidden_size, 18, num_layers=num_layers).to(device)
model.load_state_dict(torch.load(path))
model.eval()

SimpleRNN(
  (rnn): RNN(57, 256, num_layers=4, batch_first=True)
  (fc): Linear(in_features=256, out_features=18, bias=True)
)

In [22]:
max_length = 19
def preprocess_input(name):
    tensor = line_to_tensor(unicode_to_ascii(name))
    if tensor.size(0) < max_length:
        pad_size = max_length - tensor.size(0)
        tensor = F.pad(tensor, (0, 0, 0, pad_size))
    tensor = tensor.unsqueeze(0)
    # tensor = tensor.unsqueeze(0)
    return tensor.to(device)



def predict(name, top_n=3):
    model.eval()
    with torch.no_grad():
        input_tensor = preprocess_input(name)
        output = model(input_tensor)
        probabilities = F.softmax(output, dim=1)
        top_n_values, top_n_indices = torch.topk(probabilities, top_n, dim=1)
        top_n_indices = top_n_indices.cpu().numpy().flatten()
        top_n_labels = [list(label_to_index_dict.keys())[list(label_to_index_dict.values()).index(idx)] for idx in top_n_indices]
    return top_n_labels, top_n_values.cpu().numpy().flatten()

In [24]:

def prediction(input,n):
    
    top_n_predictions, top_n_scores = predict(input, n)
    for label, score in zip(top_n_predictions, top_n_scores):
        print(f"Predicted category: '{label}', Percentage: {score*100:.1f}")

name = "Apeldoorn"
prediction(name,3)


Predicted category: 'English', Percentage: 65.3
Predicted category: 'Dutch', Percentage: 23.8
Predicted category: 'Irish', Percentage: 4.9
